# Computational Workflows for biomedical data

Welcome to the course Computational Workflows for Biomedical Data. Over the next two weeks, you will learn how to leverage nf-core pipelines to analyze biomedical data and gain hands-on experience in creating your own pipelines, with a strong emphasis on Nextflow and nf-core.

Course Structure:

- Week 1: You will use a variety of nf-core pipelines to analyze a publicly available biomedical study.
- Week 2: We will shift focus to learning the basics of Nextflow, enabling you to design and implement your own computational workflows.<br>
- Final Project: The last couple of days, you will apply your knowledge to create a custom pipeline for analyzing biomedical data using Nextflow and the nf-core template.

## Basics

If you have not installed all required software, please do so now asap!


If you already installed all software, please go on and start answering the questions in this notebook. If you have any questions, don't hesitate to approach us.

1. What is nf-core?

Nf-core is a global, open-source community that uses the Nextflow workflow language to build peer-reviewed, standardized bioinformatics pipelines. It offers modular, portable and reproducible workflows for research.

2. How many pipelines are there currently in nf-core?

157 (101 released, 43 under development, 13 archived)

3. Are there any non-bioinformatic pipelines in nf-core?

Nf-core focuses on bioinformatic pipelines for analysing biological data, but since it's an open-sourced, general workflow manager,
it's not limited to bioinformatical tasks.

4. Let's go back a couple of steps. What is a pipeline and what do we use it for?

A pipeline is a sequence of processing steps where the output of one step is fed as input to the next step. Pipelines are used for organizing and automating tasks in linear or branching systems, ensuring consistent and reliable results for each step.

5. Why do you think nf-core adheres to strict guidelines?

Because the reproducability of scientific studies (especially about cancer research) is a consistent problem in science, since programs or systems get updated or simply aren't maintained, making it hard to reproduce processes.

6. What are the main features of nf-core pipelines?

The nf-core pipelines all have a identical command structure and support extensive configuration options to match the computer's infrastructure, resource requirements and reference data needs of each user.

## Let's start using the pipelines

1. Find the nf-core pipeline used to measure differential abundance of genes

In [ ]:
# run the pipeline in a cell 
# to run bash in jupyter notebooks, simply use ! before the command
# e.g.

!pwd

# For the tasks in the first week, please use the command line to run your commands and simply paste the commands you used in the respective cells!


### Local configuration

A `.wslconfig` file was used to increase the memory available to WSL, as the
default available memory was insufficient for one of the pipeline processes.

An empty Docker configuration was additionally used as a local workaround for
a Docker credential-helper issue in WSL when pulling containers from the
Seqera container registry. Therefore, some pipeline runs were executed with
`DOCKER_CONFIG=/tmp/docker-test`.

In [ ]:
# run the pipeline in the test profile using docker containers
# make sure to specify the version you want to use (use the latest one)

!nextflow run nf-core/differentialabundance -r 2.0.0 -profile test,docker --outdir test_results

In [ ]:
# repeat the run. What did change?

!nextflow run nf-core/differentialabundance -r 2.0.0 -profile test,docker --outdir test_results

# Repeating the run without -resume executes all processes again.
# In the repeated run, all 25 tasks succeeded and none were cached.
# The run took 7m 7s.

In [ ]:
# now set -resume to the command. What did change?

!nextflow run nf-core/differentialabundance -r 2.0.0 -profile test,docker --outdir test_results -resume

# With -resume, Nextflow reused successfully completed tasks from the previous run
# instead of executing them again. Many processes were therefore shown as "cached".

Check out the current directory. Next to the outdir you specified, what else has changed?

In [ ]:
!ls -la

# Besides the specified output directory "test_results", Nextflow created a
# "work" directory, a ".nextflow" directory, and ".nextflow.log" files.
# The work directory contains intermediate files generated by pipeline processes,
# while the .nextflow files/directories store information related to the Nextflow runs.

In [ ]:
# delete the work directory and run the pipeline again using -resume. What did change?

!rm -rf work

# After deleting the work directory, all 25 tasks were executed again,
# even though -resume was used.
# The cached results could no longer be reused because the corresponding
# files in the work directory had been deleted.

What changed?

Comparison of pipeline runs:

| Run | `-resume` | `work/` available | Executed | Cached | Duration | Observation |
|---|---|---|---:|---:|---|---|
| Repeated run | No | Yes | 25 | 0 | 7m 10s | All 25 tasks were executed again. |
| Run with `-resume` | Yes | Yes | 0 | 25 | 28.1s | All 25 tasks were reused from the cache. |
| After deleting `work/` | Yes | No | 25 | 0 | 6m 9s | All 25 tasks had to be executed again because the cached work files were no longer available. |

Using `-resume` allowed Nextflow to reuse all 25 previously completed tasks, reducing the runtime from several minutes to only 28.1 seconds.
After deleting the `work` directory, the cached task outputs were no longer available and all 25 tasks had to be executed again despite using `-resume`.

## Lets look at the results

### What is differential abundance analysis?

Give the most important plots from the report:

#### PCA plot

<img src="../test_results/plots/exploratory/rnaseq_deseq2_gsea/treatment/png/pca2d.png" width="500">

The PCA plot shows the overall similarity between samples based on gene expression.
PC1 explains 45.3% and PC2 22.1% of the variance. The treatment groups show
some separation, although the separation is not complete.

#### Volcano plot

<img src="../test_results/plots/differential/rnaseq_deseq2_gsea/treatment_mCherry_hND6__SRP254919/png/volcano.png" width="500">

The volcano plot shows differential gene expression by plotting log(2) fold change
against the adjusted p-value. Only a few genes pass the specified thresholds
$(|log(2) \text{ FoldChange}| > 2 \text{ and} \text{ padj} < 0.05)$.

#### DESeq2 dispersion plot

<img src="../test_results/plots/qc/rnaseq_deseq2_gsea/treatment_mCherry_hND6__SRP254919.deseq2.dispersion.png" width="500">

The dispersion plot shows the relationship between mean normalized gene expression
and dispersion. Gene-wise dispersion estimates are fitted to an overall trend,
which is used to obtain the final dispersion estimates.